In [46]:
import cmsisdsp as dsp
import os

os.environ.setdefault("MPLCONFIGDIR", ".matplotlib")

from IPython.display import display, Markdown, Math
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

def np_to_latex_matrix(name, M, precision=4):
    rows = []
    for row in M:
        row_text = " & ".join(f"{value:.{precision}g}" for value in row)
        rows.append(row_text)

    body = r" \\ ".join(rows)

    return rf"{name} = \begin{{bmatrix}} {body} \end{{bmatrix}}"

# Buck converter parameters

| Parameter | Value | Description | Unit |
| --- | ---: | --- | --- |
| `R_L` | `10e-3` | Inductor resistance | Ohm |
| `L` | `5e-4` | Inductance | H |
| `C` | `5.8e-4` | Capacitance | F |
| `f_sw` | `10e3` | Switching frequency | Hz |
| `V_in` | `13` | Input voltage | V |
| `V_out` | `5` | Output voltage | V |
| `R` | `3.5` | Load resistance | Ohm |
| `mu` | `8e3` | Observer gain | - |

In [47]:
# Buck converter parameters
R_L     = np.float32(10e-3)     # Inductor resistance   [Ohm]
L       = np.float32(5e-4)      # Inductance            [H]
C       = np.float32(5.8e-4)    # Capacitance           [F]
f_sw    = np.float32(10e3)      # Switching frequency   [Hz]
V_in    = np.float32(13)        # Input voltage         [V]
V_out   = np.float32(5)         # Output voltage        [V]
R       = np.float32(3.5)       # Load resistance       [Ohm]
mu      = np.float32(8e3)       # Observer gain         [-]

$$
A = 
    \begin{bmatrix}
        -\frac{R_L}{L} & -\frac{1}{L} \\
        \frac{1}{C} & 0
    \end{bmatrix}
$$

$$
B_0 = 
    \begin{bmatrix}
        0 & 0 \\
        0 & -\frac{1}{C}
    \end{bmatrix}
$$

$$
B_1 = 
    \begin{bmatrix}
        \frac{1}{L} & 0 \\
        0 & -\frac{1}{C}
    \end{bmatrix}
$$

$$
C = 
    \begin{bmatrix}
        1 & 1
    \end{bmatrix}
$$

In [48]:
A_c = np.array([
    [-R_L / L, -1 / L],
    [1 / C, 0]
    ], dtype=np.float32)
B_0c = np.array([
    [0, 0],
    [0, -1 / C]
    ], dtype=np.float32)
B_1c = np.array([
    [1 / L, 0],
    [0, -1 / C]
    ], dtype=np.float32)
C_c = np.array([
    [1, 1]
    ], dtype=np.float32)
D_c = np.array([[0]], dtype=np.float32)

display(Math(np_to_latex_matrix("A_c", A_c, 4)))
display(Math(np_to_latex_matrix("B_{0c}", B_0c, 4)))
display(Math(np_to_latex_matrix("B_{1c}", B_1c, 4)))
display(Math(np_to_latex_matrix("C_c", C_c, 4)))
display(Math(np_to_latex_matrix("D_c", D_c, 4)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
T_s = np.float32(1 / f_sw / 1e2)

A_0d, B_0d, C_0d, D_0d, _ = signal.cont2discrete((A_c, B_0c, C_c, D_c), T_s, method='zoh')
A_0d = A_0d.astype(np.float32)
B_0d = B_0d.astype(np.float32)
C_0d = C_0d.astype(np.float32)
D_0d = D_0d.astype(np.float32)

print("Continuous-time state-space matrices:")
print("A_0d:\n", A_0d)
display(Math(np_to_latex_matrix("A_{0d}", A_0d, 6)))
print("B_0d:\n", B_0d)
display(Math(np_to_latex_matrix("B_{0d}", B_0d, 6)))
print("C_0d:\n", C_0d)
display(Math(np_to_latex_matrix("C_{0d}", C_0d, 6)))
print("D_0d:\n", D_0d)
display(Math(np_to_latex_matrix("D_{0d}", D_0d, 6)))

Continuous-time state-space matrices:


<IPython.core.display.Math object>

A_0d:
 [[ 0.9999783  -0.00199998]
 [ 0.00172412  0.9999983 ]]


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
A_1d, B_1d, C_1d, D_1d, _ = signal.cont2discrete((A_c, B_1c, C_c, D_c), T_s, method='zoh')
A_1d = A_1d.astype(np.float32)
B_1d = B_1d.astype(np.float32)
C_1d = C_1d.astype(np.float32)
D_1d = D_1d.astype(np.float32)

print("Continuous-time state-space matrices:")
print("A_1d:\n", A_1d)
display(Math(np_to_latex_matrix("A_{1d}", A_1d, 6)))
print("B_1d:\n", B_1d)
display(Math(np_to_latex_matrix("B_{1d}", B_1d, 6)))
print("C_1d:\n", C_1d)
display(Math(np_to_latex_matrix("C_{1d}", C_1d, 6)))
print("D_1d:\n", D_1d)
display(Math(np_to_latex_matrix("D_{1d}", D_1d, 6)))

print(np.allclose(A_0d, A_1d))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

True
